# t-SNE (t-Distributed Stochastic Neighbor Embedding)

## 1. What is t-SNE?
**t-SNE** is a powerful, unsupervised **non-linear dimensionality reduction** technique. 

Unlike PCA (which tries to preserve the global variance of the data), t-SNE is specifically designed for **data visualization**. Its primary goal is to take incredibly high-dimensional data (like images or text embeddings) and map it down to 2D or 3D space while strictly preserving the **local structure** (keeping similar points close together).



---

## 2. PCA vs. t-SNE (Why use t-SNE?)
If we already have PCA, why do we need t-SNE?

* **PCA is Linear:** It rotates and projects data using straight lines (axes). If your data lies on a complex, twisted manifold (like a Swiss Roll), PCA will crush it and ruin the clusters.
* **t-SNE is Non-Linear:** It calculates pairwise distances and can untangle highly complex, non-linear relationships.

| Feature | PCA | t-SNE |
| :--- | :--- | :--- |
| **Type** | Linear | Non-Linear |
| **Primary Goal** | Feature Extraction / Noise Reduction | Data Visualization |
| **Structure Preserved** | Global structure (Overall variance) | Local structure (Neighborhoods) |
| **Deterministic?** | Yes (Always gives the same result) | No (Stochastic/Randomized) |
| **Can transform new data?** |  Yes (`pca.transform(X_new)`) |  No (Must fit all data at once) |



---

## 3. How t-SNE Works (The Math Intuition)
t-SNE works in three main steps:

**Step 1: Measure High-Dimensional Similarities (Gaussian)**
It calculates the probability that two points are neighbors in the original, high-dimensional space. It uses a **Gaussian (Normal) distribution** to do this. If points are close, the probability is high; if they are far, the probability drops to almost zero.
$$p_{j|i} = \frac{\exp(-||x_i - x_j||^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-||x_i - x_k||^2 / 2\sigma_i^2)}$$

**Step 2: Measure Low-Dimensional Similarities (t-Distribution)**
It then randomly maps the points onto a 2D or 3D map. It measures the similarities again, but this time it uses a **Student's t-distribution** (which has "heavier tails" than a Gaussian). 
* *Why a t-distribution?* The heavy tails solve the "Crowding Problem," pushing dissimilar points further apart in the 2D space so they don't clump into a giant mess.
$$q_{ij} = \frac{(1 + ||y_i - y_j||^2)^{-1}}{\sum_{k \neq l} (1 + ||y_k - y_l||^2)^{-1}}$$

**Step 3: Minimize the Difference (KL Divergence)**
t-SNE uses gradient descent to move the points around in the 2D space. The goal is to minimize the difference between the high-dimensional probabilities ($P$) and the low-dimensional probabilities ($Q$). The math metric used for this is called **Kullback-Leibler (KL) Divergence**:
$$KL(P || Q) = \sum_i \sum_j p_{ij} \log \frac{p_{ij}}{q_{ij}}$$

---

## 4. The Most Important Parameter: Perplexity
In t-SNE, you don't choose the number of clusters. Instead, you tune **Perplexity**.

* **Definition:** Perplexity is essentially a guess about the number of close neighbors each point has. It balances attention between local and global aspects of your data.
* **Typical Range:** Usually set between **5 and 50**.
* **Rule of Thumb:** * Too low ($\sim 2$): The data forms meaningless, tiny clumps.
  * Too high ($\sim 100+$): It acts too much like PCA, merging distinct clusters.

---

## 5. Python Implementation (Scikit-Learn)

```python
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Initialize t-SNE
# Note: n_components is almost always 2 or 3 for visualization
tsne = TSNE(
    n_components=2, 
    perplexity=30.0, 
    learning_rate='auto', 
    init='pca',            # 'pca' initialization is highly recommended for stability
    random_state=42
)

# Fit and transform the data (t-SNE does this in one step)
X_tsne = tsne.fit_transform(X)

# Plotting the result
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap='viridis')
plt.title("t-SNE Visualization")
plt.show()

```

---

## 6. Crucial Limitations 

If you get asked about t-SNE in an interview, knowing its flaws is just as important as knowing how it works.
 - **Disadvantages & Traps**
   - **Cluster sizes mean nothing:** t-SNE expands dense clusters and shrinks sparse ones. A large visual cluster doesn't mean the data is more spread out.
   - **Distances between clusters mean nothing:** Just because Cluster A is physically far from Cluster B on the t-SNE plot does not mean they are very different in the original data. Global geometry is lost.
   - **Computationally Heavy:** $\mathcal{O}(n^2)$ complexity. It is extremely slow on datasets with $>100,000$ rows (though Barnes-Hut approximations help).
   - **No transform method:** You cannot train a t-SNE model and then feed it new, unseen test data. It only works on the data it was trained on.
- **Advantages**
   - Produces the most beautiful, distinctly separated 2D/3D visualizations of complex data.
   - Exceptional at handling non-linear relationships.
   - The go-to standard for visualizing deep learning embeddings (like Word2Vec or CNN image features).
